# pFBA yields (carbon-normalized)

Recalculate yields from blood-constrained pFBA fluxes with correct carbon formula.
Run from repo root after blood-constrained simulation.

In [ ]:
import re
import warnings
from pathlib import Path

import cobra
import pandas as pd

warnings.filterwarnings("ignore")

MODEL_DIR = Path("model/contextualized_models")
STAGES = ("Af", "Am")
MAX_CARBON = 50  # Exclude large molecules from denominator

print(f"Stages: {STAGES}")
print(f"Carbon threshold: {MAX_CARBON}")

In [ ]:
def carbon_count(formula):
    """Extract carbon count from metabolite formula"""
    if not formula:
        return 0.0
    match = re.search(r"C(?![a-z])(\d*(?:\.\d+)?)", str(formula))
    return float(match.group(1) or 1) if match else 0.0

def exchange_formula_map(model):
    """Get metabolite formula for each exchange reaction"""
    out = {}
    for reaction in model.exchanges:
        metabolites = list(reaction.metabolites)
        met = metabolites[0] if metabolites else None
        out[reaction.id] = (met.id if met else None, getattr(met, "formula", "") if met else "")
    return out

print("Helper functions defined")

In [ ]:
# Process each stage
for stage in STAGES:
    print(f"\n{stage}:")
    
    # Load model and flux file
    model = cobra.io.read_sbml_model(str(MODEL_DIR / f"{stage}_c.xml"))
    exch = exchange_formula_map(model)
    fluxes = pd.read_csv(f"{stage}_b_c_pfba_fluxes.csv")
    
    # Calculate carbon uptake
    rows = []
    total_carbon_consumed = 0.0
    
    for _, row in fluxes.iterrows():
        rid = row.reaction_id
        met_id, formula = exch.get(rid, (None, ""))
        n_c = carbon_count(formula)
        uptake = max(0.0, -float(row.flux)) if rid in exch else 0.0
        contribution = uptake * n_c
        excluded_large = n_c > MAX_CARBON
        
        if rid in exch and uptake > 0 and n_c > 0 and not excluded_large:
            total_carbon_consumed += contribution
        
        rows.append({
            "reaction_id": rid, "is_exchange": rid in exch,
            "metabolite_id": met_id, "formula": formula, "n_carbon": n_c,
            "flux": row.flux, "uptake": uptake, "carbon_contribution": contribution,
            "excluded_large_molecule": excluded_large,
        })
    
    rows_df = pd.DataFrame(rows)

In [ ]:
    # Show excluded large molecules
    excluded = rows_df[rows_df.is_exchange & (rows_df.uptake > 0) & (rows_df.n_carbon > MAX_CARBON)]
    if len(excluded):
        print(f"  Excluded large molecules (C > {MAX_CARBON}):")
        print(excluded[["reaction_id", "metabolite_id", "formula", "n_carbon"]].to_string(index=False))
    
    # Carbon sources (sorted by contribution)
    sources = rows_df[
        rows_df.is_exchange & (rows_df.uptake > 0) & (rows_df.n_carbon > 0) & ~rows_df.excluded_large_molecule
    ]
    sources = sources.sort_values("carbon_contribution", ascending=False)
    sources.to_csv(f"{stage}_carbon_uptake_sources.csv", index=False)
    
    print(f"  Total carbon consumed: {total_carbon_consumed:.4f} mmol-C/gDW/h")
    print(f"  Carbon sources: {len(sources)}")

In [ ]:
    # Calculate yields
    yields = fluxes.copy()
    yields["carbon_normalized_yield"] = yields["flux"] / total_carbon_consumed
    yields.to_csv(f"{stage}_pfba_yields_corrected.csv", index=False)
    
    print(f"  Saved yields and sources CSVs")

print("\n✓ Done")